# Data Cleaning & Preparation Notebook

## Research Question
**Is biodiversity loss in beef-exporting countries associated with the volume of beef exports to the United Kingdom?**

## Data Sources
| Dataset | File | Description |
|---------|------|-------------|
| BIO | `bio-raw.csv` | Biodiversity Intactness Index (BII) per country, scenario, and year. Historical: 1970–2014; SSP projections: 2015–2050. |
| FAOSTAT | `FAOSTAT_data_en_3-24-2026.csv` | Annual bilateral trade: beef & veal export quantities to the UK, by reporting country, 2010–2020. |

## Variables
| Variable | Role | Description |
|----------|------|-------------|
| `bii` | Predictor (raw) | Biodiversity Intactness Index (0–1; 1 = fully intact) |
| `bii_loss` | Predictor (derived) | 1 − BII (higher = more biodiversity loss) |
| `uk_import_qty_t` | Outcome (raw) | Beef export quantity to UK (tonnes) |
| `log_uk_import_qty_t` | Outcome (transformed) | log(1 + uk_import_qty_t) — corrects right-skew |

## Timescale Rationale
- Historical BIO data runs to **2014**; FAOSTAT starts at **2010**.
- Overlap for fully observed data: **2010–2014** (5 years).
- To extend to 2020 we stitch in **SSP2** ("middle of the road") BIO projections for 2015–2020.

## Outputs
All saved to `data/clean/`:
| File | Description |
|------|-------------|
| `bio_clean.csv` | BII per country × year × scenario, ISO3 extracted |
| `fao_clean.csv` | FAOSTAT tidy: iso3, country, year, uk_import_qty_t |
| `panel_2010_2014.csv` | Panel (country × year), 2010–2014, historical BIO only |
| `crosssection_avg.csv` | Cross-sectional country averages, 2010–2014 |
| `panel_extended_2010_2020.csv` | Extended panel: historical + SSP2 BIO, 2010–2020 |

---
## 0 · Setup

In [74]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)

# ── Paths ──────────────────────────────────────────────────────────────────
# This notebook lives in data/, so paths are relative to that folder.
DATA = os.path.dirname(os.path.abspath("__file__"))  # data/
OUT  = os.path.join(DATA, "clean")
os.makedirs(OUT, exist_ok=True)

BIO_PATH = os.path.join(DATA, "bio-raw.csv")
FAO_PATH = os.path.join(DATA, "FAOSTAT_data_en_3-24-2026.csv")

# ── Parameters ─────────────────────────────────────────────────────────────
# Minimum total exports to UK (tonnes) across all years to keep a country.
MIN_TOTAL_TRADE_T = 10

# Importing country we focus on.
IMPORT_COUNTRY = "United Kingdom of Great Britain and Northern Ireland"

# SSP scenario used to extend BIO beyond 2014.
EXTENSION_SCENARIO = "ssp2rcp4p5messageglobiom"  # "middle of the road"

print(f"BIO path:  {BIO_PATH}")
print(f"FAO path:  {FAO_PATH}")
print(f"Output dir: {OUT}")

BIO path:  /Users/mosser/PycharmProjects/sklearn-group/data/bio-raw.csv
FAO path:  /Users/mosser/PycharmProjects/sklearn-group/data/FAOSTAT_data_en_3-24-2026.csv
Output dir: /Users/mosser/PycharmProjects/sklearn-group/data/clean


### ISO-3 Country-Code Helper

We use a comprehensive built-in lookup table that covers every FAOSTAT
reporter-country name (including FAO-specific spellings like "Trkiye" and
"China, mainland").  `pycountry` is used only as a fallback for any names
not already in the table.

> **Why not rely on `pycountry` alone?**  Two reasons:
> 1. `pycountry` may not be installed in the Jupyter kernel (the old code
>    caught `ImportError` silently via `except Exception: pass`, causing
>    *all* non-override lookups to return `None`).
> 2. Some FAO names ("China, mainland", "Netherlands (Kingdom of the)",
>    "Trkiye") have no match in the ISO standard even with fuzzy search.

In [75]:
# ── Complete FAOSTAT-name → ISO3 lookup ────────────────────────────────────
# Covers every Reporter Countries value in the FAOSTAT beef-trade extract.
FAOSTAT_ISO3 = {
    "Argentina":                    "ARG",
    "Australia":                    "AUS",
    "Austria":                      "AUT",
    "Bahrain":                      "BHR",
    "Belgium":                      "BEL",
    "Botswana":                     "BWA",
    "Brazil":                       "BRA",
    "Brunei Darussalam":            "BRN",
    "Bulgaria":                     "BGR",
    "China, mainland":              "CHN",
    "Croatia":                      "HRV",
    "Czechia":                      "CZE",
    "Denmark":                      "DNK",
    "Dominican Republic":           "DOM",
    "Estonia":                      "EST",
    "Finland":                      "FIN",
    "France":                       "FRA",
    "Germany":                      "DEU",
    "Greece":                       "GRC",
    "Hungary":                      "HUN",
    "Iceland":                      "ISL",
    "Ireland":                      "IRL",
    "Italy":                        "ITA",
    "Jamaica":                      "JAM",
    "Latvia":                       "LVA",
    "Lebanon":                      "LBN",
    "Lithuania":                    "LTU",
    "Luxembourg":                   "LUX",
    "Malta":                        "MLT",
    "Mozambique":                   "MOZ",
    "Namibia":                      "NAM",
    "Netherlands (Kingdom of the)": "NLD",
    "New Zealand":                  "NZL",
    "Norway":                       "NOR",
    "Philippines":                  "PHL",
    "Poland":                       "POL",
    "Portugal":                     "PRT",
    "Republic of Korea":            "KOR",
    "Romania":                      "ROU",
    "Singapore":                    "SGP",
    "Slovakia":                     "SVK",
    "Slovenia":                     "SVN",
    "South Africa":                 "ZAF",
    "Spain":                        "ESP",
    "Sweden":                       "SWE",
    "Switzerland":                  "CHE",
    "Trinidad and Tobago":          "TTO",
    "Trkiye":                       "TUR",
    "Türkiye":                      "TUR",
    "Ukraine":                      "UKR",
    "United Arab Emirates":         "ARE",
    "United States of America":     "USA",
    "Uruguay":                      "URY",
    "Viet Nam":                     "VNM",
}

# Track whether pycountry is available (checked once, not per-row).
try:
    import pycountry as _pycountry
    _HAS_PYCOUNTRY = True
except ImportError:
    _HAS_PYCOUNTRY = False
    print("NOTE: pycountry is not installed — using built-in FAOSTAT_ISO3 table only.")


def name_to_iso3(name: str):
    """Return ISO 3166-1 alpha-3 for a country name string.

    Resolution order:
      1. Built-in FAOSTAT_ISO3 dictionary  (fast, no dependencies)
      2. pycountry exact match             (if installed)
      3. pycountry fuzzy search            (if installed)
    """
    # 1) Built-in table — covers all known FAOSTAT names
    if name in FAOSTAT_ISO3:
        return FAOSTAT_ISO3[name]

    # 2) pycountry fallback for any new / unknown names
    if _HAS_PYCOUNTRY:
        try:
            result = _pycountry.countries.get(name=name)
            if result:
                return result.alpha_3
            hits = _pycountry.countries.search_fuzzy(name)
            if hits:
                return hits[0].alpha_3
        except LookupError:
            pass

    return None

---
# Part A — Biodiversity (BIO) Data
## 1 · Load & Explore Raw BIO Data

The BIO dataset contains multiple indicators (BII, land-use shares, etc.).
Since our research question focuses on **biodiversity loss**, we retain only
the **Biodiversity Intactness Index (`bii`)** and derive `bii_loss = 1 − bii`.
All other indicators (pastureland, crops, highintensityag, etc.) are dropped.

In [76]:
bio_raw = pd.read_csv(BIO_PATH, low_memory=False)

print(f"Shape: {bio_raw.shape}")
print(f"Columns: {list(bio_raw.columns)}")
print(f"Year range: {bio_raw['year'].min()} – {bio_raw['year'].max()}  (dtype: {bio_raw['year'].dtype})")
print(f"Scenarios: {bio_raw['scenario'].unique()}")
print(f"Variables (indicators): {bio_raw['variable'].unique()}")
bio_raw.head()

Shape: (376992, 8)
Columns: ['_id', 'area_code', 'lower_uncertainty', 'scenario', 'upper_uncertainty', 'value', 'variable', 'year']
Year range: 1970 – 2050  (dtype: int64)
Scenarios: <StringArray>
[          'ssp4rcp6p0gcam',   'ssp5rcp8p5remindmagpie',               'historical',          'ssp1rcp2p6image',
 'ssp2rcp4p5messageglobiom',            'ssp3rcp7p0aim']
Length: 6, dtype: str
Variables (indicators): <StringArray>
['bii', 'crops', 'highintensityag', 'hpd', 'pastureland', 'qualitynatural', 'urbanextent']
Length: 7, dtype: str


,_id,area_code,lower_uncertainty,scenario,upper_uncertainty,value,variable,year
0,344002,001-002-202-018-BWA,NaN,ssp4rcp6p0gcam,NaN,0.759763,bii,2044
1,344003,001-002-202-018-LSO,NaN,ssp4rcp6p0gcam,NaN,0.553173,bii,2044
2,344006,001-002-202-018-ZAF,NaN,ssp4rcp6p0gcam,NaN,0.623727,bii,2044
3,344014,001-009-054-PNG,NaN,ssp4rcp6p0gcam,NaN,0.986023,bii,2044
4,344023,001-009-057-PLW,NaN,ssp4rcp6p0gcam,NaN,NaN,bii,2044


### 1.1 · Validate: null counts & data quality

In [77]:
print("Null counts per column:")
print(bio_raw.isna().sum())
print(f"\nRows with null area_code: {bio_raw['area_code'].isna().sum():,}")
print(f"Rows with null value:     {bio_raw['value'].isna().sum():,}")
print(f"Total rows:               {len(bio_raw):,}")

Null counts per column:
_id                       0
area_code              1386
lower_uncertainty    371217
scenario                  0
upper_uncertainty    371217
value                 38412
variable                  0
year                      0
dtype: int64

Rows with null area_code: 1,386
Rows with null value:     38,412
Total rows:               376,992


### 1.2 · Understand the `area_code` structure
Each `area_code` is a dash-separated hierarchy (continent-region-…-ISO3).
The **last segment** is the ISO 3166-1 alpha-3 country code — that's what we need.

In [78]:
print("Sample area_codes:")
print(bio_raw["area_code"].dropna().unique()[:10])

Sample area_codes:
<StringArray>
['001-002-202-018-BWA', '001-002-202-018-LSO', '001-002-202-018-ZAF',     '001-009-054-PNG',     '001-009-057-PLW',
     '001-009-061-COK',     '001-009-061-NIU',     '001-009-061-PYF', '001-019-419-005-COL', '001-019-419-005-ECU']
Length: 10, dtype: str


## 2 · Clean BIO Data

Steps:
1. **Drop rows** with null `area_code` (no country identity → useless).
2. **Extract ISO3** from the last segment of `area_code`.
3. **Keep only valid 3-letter codes** (drop supra-national aggregates like numeric codes).
4. **Filter to `bii` indicator only** — the sole predictor for our research question.
5. **Drop unneeded columns**: `_id` (database artefact), `lower_uncertainty` / `upper_uncertainty` (all NA for BII).
6. **Drop rows with null `value`** (no measurement → can't model).
7. **Pivot wide** so `bii` becomes its own column.
8. **Derive `bii_loss`** = 1 − BII (higher = more degraded, easier to interpret as a predictor).
9. **Drop countries with no BII data** — small islands/territories that lack BII and don't export beef to the UK.

In [79]:
# Step 2.1 – Drop rows with no area code
bio = bio_raw.dropna(subset=["area_code"]).copy()
print(f"After dropping null area_code: {bio.shape[0]:,} rows  (removed {len(bio_raw) - len(bio):,})")

# Step 2.2 – Extract ISO3 from the last segment of area_code
bio["iso3"] = bio["area_code"].str.split("-").str[-1].str.upper()

# Step 2.3 – Keep only standard 3-letter ISO3 codes
bio = bio[bio["iso3"].str.match(r"^[A-Z]{3}$", na=False)]
print(f"After filtering to valid ISO3 codes: {bio.shape[0]:,} rows  |  {bio['iso3'].nunique()} countries")

# Step 2.4 – Keep only the BII indicator (our sole biodiversity predictor)
bio = bio[bio["variable"] == "bii"].copy()
print(f"After filtering to BII indicator: {bio.shape[0]:,} rows")

# Step 2.5 – Drop columns not needed for modelling
bio.drop(columns=["_id", "area_code", "lower_uncertainty", "upper_uncertainty", "variable"], inplace=True)

# Step 2.6 – Drop rows with missing BII value
before = len(bio)
bio.dropna(subset=["value"], inplace=True)
print(f"After dropping null values: {bio.shape[0]:,} rows  (removed {before - len(bio):,})")

# Step 2.7 – Rename value column
bio.rename(columns={"value": "bii"}, inplace=True)

# Step 2.8 – Derive bii_loss = 1 - bii  (higher = more degraded)
bio["bii_loss"] = 1 - bio["bii"]

INDICATOR_COLS = ["bii", "bii_loss"]

print(f"\nCleaned BIO: {bio.shape}")
print(f"  Scenarios:  {bio['scenario'].nunique()} — {sorted(bio['scenario'].unique())}")
print(f"  Countries:  {bio['iso3'].nunique()}")
print(f"  Years:      {bio['year'].min()} – {bio['year'].max()}")
bio.head()

After dropping null area_code: 375,606 rows  (removed 1,386)
After filtering to valid ISO3 codes: 332,640 rows  |  240 countries
After filtering to BII indicator: 47,520 rows
After dropping null values: 39,798 rows  (removed 7,722)

Cleaned BIO: (39798, 5)
  Scenarios:  6 — ['historical', 'ssp1rcp2p6image', 'ssp2rcp4p5messageglobiom', 'ssp3rcp7p0aim', 'ssp4rcp6p0gcam', 'ssp5rcp8p5remindmagpie']
  Countries:  201
  Years:      1970 – 2050


,scenario,bii,year,iso3,bii_loss
0,ssp4rcp6p0gcam,0.759763,2044,BWA,0.240237
1,ssp4rcp6p0gcam,0.553173,2044,LSO,0.446827
2,ssp4rcp6p0gcam,0.623727,2044,ZAF,0.376273
3,ssp4rcp6p0gcam,0.986023,2044,PNG,0.013977
8,ssp4rcp6p0gcam,0.716595,2044,COL,0.283405


In [80]:
# Data is already in wide format (one row per iso3 × year × scenario)
# since we filtered to the single BII indicator above.

bio_wide = bio.copy()

print(f"BIO shape: {bio_wide.shape}")
print(f"Columns: {list(bio_wide.columns)}")
print(f"Countries: {bio_wide['iso3'].nunique()}")
bio_wide.head(10)

BIO shape: (39798, 5)
Columns: ['scenario', 'bii', 'year', 'iso3', 'bii_loss']
Countries: 201


,scenario,bii,year,iso3,bii_loss
0,ssp4rcp6p0gcam,0.759763,2044,BWA,0.240237
1,ssp4rcp6p0gcam,0.553173,2044,LSO,0.446827
2,ssp4rcp6p0gcam,0.623727,2044,ZAF,0.376273
3,ssp4rcp6p0gcam,0.986023,2044,PNG,0.013977
8,ssp4rcp6p0gcam,0.716595,2044,COL,0.283405
9,ssp4rcp6p0gcam,0.789611,2044,ECU,0.210389
10,ssp4rcp6p0gcam,0.893540,2044,PER,0.106460
11,ssp4rcp6p0gcam,0.313319,2044,URY,0.686681
12,ssp4rcp6p0gcam,0.856686,2044,BLZ,0.143314
13,ssp4rcp6p0gcam,0.616668,2044,NIC,0.383332


### 2.1 · Validate: BIO cleaning results

In [81]:
print("Null counts per column after cleaning:")
print(bio_wide[INDICATOR_COLS].isna().sum())

print(f"\nHistorical rows: {(bio_wide['scenario'] == 'historical').sum():,}")
print(f"SSP2 rows:       {(bio_wide['scenario'] == EXTENSION_SCENARIO).sum():,}")

print("\nDescriptive stats — historical scenario:")
bio_wide[bio_wide["scenario"] == "historical"][INDICATOR_COLS].describe().round(4)

Null counts per column after cleaning:
bii         0
bii_loss    0
dtype: int64

Historical rows: 3,618
SSP2 rows:       7,236

Descriptive stats — historical scenario:


,bii,bii_loss
count,3618.0000,3618.0000
mean,0.7622,0.2378
std,0.1597,0.1597
min,0.2982,0.0000
25%,0.6377,0.0958
50%,0.7623,0.2377
75%,0.9042,0.3623
max,1.0000,0.7018


In [82]:
# Save cleaned BIO
bio_wide.to_csv(os.path.join(OUT, "bio_clean.csv"), index=False)
print(f"Saved → data/clean/bio_clean.csv  ({bio_wide.shape[0]:,} rows, {bio_wide.shape[1]} cols)")

Saved → data/clean/bio_clean.csv  (39,798 rows, 5 cols)


### 2.2 · Historical BIO slice (for merging later)
We isolate the **historical** scenario — these are observed data (not model projections).
This is the most reliable subset for analysis.

In [83]:
bio_hist = (
    bio_wide[bio_wide["scenario"] == "historical"]
    .drop(columns=["scenario"])
    .copy()
)
print(f"Historical BIO: {bio_hist.shape[0]:,} rows")
print(f"Year range: {bio_hist['year'].min()} – {bio_hist['year'].max()}")
print(f"Countries:  {bio_hist['iso3'].nunique()}")
bio_hist.head()

Historical BIO: 3,618 rows
Year range: 1970 – 2014
Countries:  201


,bii,year,iso3,bii_loss
283989,0.966643,1970,DZA,0.033357
283990,1.000000,1970,EGY,0.000000
283991,0.943249,1970,BFA,0.056751
283992,0.912323,1970,MLI,0.087677
283993,0.846239,1970,MRT,0.153761


---
# Part B — FAOSTAT Trade Data
## 3 · Load & Explore Raw FAOSTAT Data

> **Note:** After cleaning, only ~32 countries remain with non-negligible
> UK beef exports. Trade volumes are heavily concentrated in a few countries
> (Ireland, Brazil, Netherlands) — log-transformation is used to reduce
> skewness and the influence of dominant exporters.

In [84]:
fao_raw = pd.read_csv(FAO_PATH, encoding="utf-8-sig")

print(f"Shape: {fao_raw.shape}")
print(f"Columns: {list(fao_raw.columns)}")
print(f"Year range: {fao_raw['Year'].min()} – {fao_raw['Year'].max()}")
print(f"Partner countries: {fao_raw['Partner Countries'].unique()}")
print(f"Items: {fao_raw['Item'].unique()}")
print(f"Elements: {fao_raw['Element'].unique()}")
print(f"Reporter countries (exporters): {fao_raw['Reporter Countries'].nunique()}")
fao_raw.head()

Shape: (359, 16)
Columns: ['Domain Code', 'Domain', 'Reporter Country Code (M49)', 'Reporter Countries', 'Partner Country Code (M49)', 'Partner Countries', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description']
Year range: 2010 – 2020
Partner countries: <StringArray>
['United Kingdom of Great Britain and Northern Ireland']
Length: 1, dtype: str
Items: <StringArray>
['Beef and veal preparations nes']
Length: 1, dtype: str
Elements: <StringArray>
['Export quantity']
Length: 1, dtype: str
Reporter countries (exporters): 53


,Domain Code,Domain,Reporter Country Code (M49),Reporter Countries,Partner Country Code (M49),Partner Countries,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description
0,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2010,2010,t,1959.0,A,Official figure
1,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2011,2011,t,1322.0,A,Official figure
2,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2012,2012,t,1182.0,A,Official figure
3,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2013,2013,t,314.0,A,Official figure
4,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2014,2014,t,58.4,A,Official figure


### 3.1 · Validate: FAOSTAT data quality

In [85]:
print("Null counts:")
print(fao_raw.isna().sum())
print(f"\nFlag descriptions (data quality):")
print(fao_raw["Flag Description"].value_counts())
print(f"\nAll rows import to: {fao_raw['Partner Countries'].unique()}")

Null counts:
Domain Code                    0
Domain                         0
Reporter Country Code (M49)    0
Reporter Countries             0
Partner Country Code (M49)     0
Partner Countries              0
Element Code                   0
Element                        0
Item Code (CPC)                0
Item                           0
Year Code                      0
Year                           0
Unit                           0
Value                          0
Flag                           0
Flag Description               0
dtype: int64

Flag descriptions (data quality):
Flag Description
Official figure                        357
Value imputed by a receiving agency      2
Name: count, dtype: int64

All rows import to: <StringArray>
['United Kingdom of Great Britain and Northern Ireland']
Length: 1, dtype: str


## 4 · Clean FAOSTAT Data

Steps:
1. **Filter** for the target importing country (United Kingdom).
2. **Select & rename** columns for clarity.
3. **Drop zero-trade rows** (no useful information).
4. **Map country names → ISO3** using built-in lookup + `pycountry` fallback.
5. **Remove negligible trade partners** (< 10 tonnes total across all years).
6. **Log-transform** the outcome variable (trade volumes are heavily right-skewed).

In [86]:
# Step 4.1 – Filter for UK imports
fao = fao_raw[fao_raw["Partner Countries"] == IMPORT_COUNTRY].copy()
print(f"After filtering to UK imports: {fao.shape[0]} rows")

# Step 4.2 – Select and rename columns
fao = fao[[
    "Reporter Countries", "Partner Countries",
    "Item", "Element", "Year", "Unit", "Value", "Flag Description",
]].rename(columns={
    "Reporter Countries": "country",
    "Partner Countries":  "import_country",
    "Item":               "commodity",
    "Element":            "measure",
    "Year":               "year",
    "Unit":               "unit",
    "Value":              "uk_import_qty_t",
    "Flag Description":   "data_quality",
})

# Step 4.3 – Drop zero-trade rows
before = len(fao)
fao = fao[fao["uk_import_qty_t"] > 0].copy()
print(f"After dropping zero-trade rows: {fao.shape[0]} rows  (removed {before - len(fao)})")

# Step 4.4 – Map country names to ISO3
fao["iso3"] = fao["country"].map(name_to_iso3)

unmapped = fao[fao["iso3"].isna()]["country"].unique()
if len(unmapped):
    print(f"WARNING: Could not map to ISO3: {unmapped}")
else:
    print(f"All countries mapped to ISO3 ✓  ({fao['iso3'].nunique()} unique countries, {len(fao)} rows)")

# Show a sample from different countries (head() only shows Argentina because it's first alphabetically)
fao.groupby("country").head(1).head(10)

After filtering to UK imports: 359 rows
After dropping zero-trade rows: 328 rows  (removed 31)
All countries mapped to ISO3 ✓  (49 unique countries, 328 rows)


,country,import_country,commodity,measure,year,unit,uk_import_qty_t,data_quality,iso3
0,Argentina,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,1959.00,Official figure,ARG
7,Australia,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,7.00,Official figure,AUS
11,Austria,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2011,t,17.00,Official figure,AUT
22,Bahrain,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2014,t,0.17,Official figure,BHR
27,Belgium,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,1140.00,Official figure,BEL
38,Botswana,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,55.00,Official figure,BWA
39,Brazil,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,42473.00,Official figure,BRA
51,Bulgaria,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,2.00,Official figure,BGR
62,"China, mainland",United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2013,t,1.00,Official figure,CHN
64,Croatia,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,11.00,Official figure,HRV


In [87]:
# Step 4.5 – Remove negligible trade partners
total_by_country = fao.groupby("country")["uk_import_qty_t"].sum()
keep_countries   = total_by_country[total_by_country >= MIN_TOTAL_TRADE_T].index
removed_countries = set(fao["country"].unique()) - set(keep_countries)

fao = fao[fao["country"].isin(keep_countries)].copy()
print(f"Removed {len(removed_countries)} countries with < {MIN_TOTAL_TRADE_T}t total trade: {removed_countries}")
print(f"Remaining: {fao['country'].nunique()} exporting countries")

# Sort and reset index
fao.sort_values(["iso3", "year"], inplace=True)
fao.reset_index(drop=True, inplace=True)

# Step 4.6 – Log-transform outcome (handles right-skew)
fao["log_uk_import_qty_t"] = np.log1p(fao["uk_import_qty_t"])

print(f"\nCleaned FAOSTAT: {fao.shape}")
print(f"Year range: {fao['year'].min()} – {fao['year'].max()}")
print(f"Unique exporters: {fao['iso3'].nunique()}")

Removed 17 countries with < 10t total trade: {'Trinidad and Tobago', 'South Africa', 'Jamaica', 'Mozambique', 'Republic of Korea', 'Bahrain', 'Lebanon', 'Finland', 'Malta', 'Slovenia', 'United Arab Emirates', 'Philippines', 'Dominican Republic', 'Switzerland', 'China, mainland', 'Türkiye', 'Viet Nam'}
Remaining: 32 exporting countries

Cleaned FAOSTAT: (292, 10)
Year range: 2010 – 2020
Unique exporters: 32


### 4.1 · Validate: top exporters to the UK

In [88]:
top_exporters = (
    fao.groupby(["iso3", "country"])["uk_import_qty_t"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
print("Top 15 beef exporters to the UK (total tonnes, 2010-2020):")
print(top_exporters.to_string())

Top 15 beef exporters to the UK (total tonnes, 2010-2020):
iso3  country                     
IRL   Ireland                         403907.16
BRA   Brazil                          312350.67
FRA   France                           31069.19
DNK   Denmark                          26842.29
NLD   Netherlands (Kingdom of the)     15564.47
BEL   Belgium                          12094.29
DEU   Germany                          11391.61
POL   Poland                            9353.69
ROU   Romania                           4969.88
ARG   Argentina                         4882.10
SWE   Sweden                            4849.55
URY   Uruguay                           2945.08
ITA   Italy                             2132.19
ESP   Spain                              889.66
USA   United States of America           648.95


In [89]:
# Save cleaned FAOSTAT
fao.to_csv(os.path.join(OUT, "fao_clean.csv"), index=False)
print(f"Saved -> data/clean/fao_clean.csv  ({fao.shape[0]:,} rows)")

Saved -> data/clean/fao_clean.csv  (292 rows)


---
# Part C — Merge & Build Analysis-Ready Datasets

We join BIO indicators with FAOSTAT trade data on `(iso3, year)` to create
panel and cross-sectional datasets ready for regression.

## 5 · Panel-Building Helper

In [90]:
def build_panel(bio_slice, fao_df, label):
    # Aggregate FAOSTAT by (iso3, year, country) in case of multiple commodity rows
    fao_agg = (
        fao_df.groupby(["iso3", "year", "country"], as_index=False)
              .agg(uk_import_qty_t=("uk_import_qty_t", "sum"))
    )
    fao_agg["log_uk_import_qty_t"] = np.log1p(fao_agg["uk_import_qty_t"])

    # Inner join: only country-years present in BOTH datasets
    merged = bio_slice.merge(fao_agg, on=["iso3", "year"], how="inner")
    merged.sort_values(["iso3", "year"], inplace=True)
    merged.reset_index(drop=True, inplace=True)

    # Drop rows with NaN in any model column
    model_cols = INDICATOR_COLS + ["uk_import_qty_t"]
    before = len(merged)
    merged.dropna(subset=model_cols, inplace=True)
    dropped = before - len(merged)

    # Reorder columns: identifiers -> outcome -> predictors
    id_cols = ["iso3", "country", "year"]
    y_cols  = ["uk_import_qty_t", "log_uk_import_qty_t"]
    merged  = merged[id_cols + y_cols + INDICATOR_COLS]

    n_countries = merged["iso3"].nunique()
    n_years     = merged["year"].nunique()
    print(f"\n{label}")
    print(f"  Shape: {merged.shape}  |  {n_countries} countries x {n_years} years")
    if dropped:
        print(f"  Dropped {dropped} rows with NaN in model columns")
    return merged

## 6 · Panel 2010–2014 (Historical BIO Only)
This is the **primary analysis dataset** — both BIO and FAOSTAT are fully observed (no projections).

In [91]:
panel_hist = build_panel(
    bio_hist[bio_hist["year"].between(2010, 2014)],
    fao[fao["year"].between(2010, 2014)],
    "Panel 2010-2014 (historical BIO)",
)

panel_hist.to_csv(os.path.join(OUT, "panel_2010_2014.csv"), index=False)
print(f"Saved -> data/clean/panel_2010_2014.csv")
panel_hist.head(10)


Panel 2010-2014 (historical BIO)
  Shape: (134, 7)  |  32 countries x 5 years
Saved -> data/clean/panel_2010_2014.csv


,iso3,country,year,uk_import_qty_t,log_uk_import_qty_t,bii,bii_loss
0,ARG,Argentina,2010,1959.0,7.580700,0.725217,0.274783
1,ARG,Argentina,2011,1322.0,7.187657,0.723646,0.276354
2,ARG,Argentina,2012,1182.0,7.075809,0.721324,0.278676
3,ARG,Argentina,2013,314.0,5.752573,0.715945,0.284055
4,ARG,Argentina,2014,58.4,4.084294,0.711267,0.288733
5,AUS,Australia,2010,7.0,2.079442,0.699317,0.300683
6,AUS,Australia,2011,13.0,2.639057,0.693440,0.306560
7,AUS,Australia,2012,3.0,1.386294,0.694919,0.305081
8,AUT,Austria,2011,17.0,2.890372,0.721503,0.278497
9,AUT,Austria,2012,138.0,4.934474,0.722662,0.277338


### 6.1 · Validate: panel coverage

In [92]:
print(f"Countries in panel: {sorted(panel_hist['iso3'].unique())}")
print(f"Years in panel:     {sorted(panel_hist['year'].unique())}")
print(f"\nObservations per year:")
print(panel_hist.groupby("year")["iso3"].count())

Countries in panel: ['ARG', 'AUS', 'AUT', 'BEL', 'BGR', 'BRA', 'BWA', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LVA', 'NAM', 'NLD', 'NOR', 'NZL', 'POL', 'PRT', 'ROU', 'SVK', 'SWE', 'UKR', 'URY', 'USA']
Years in panel:     [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014)]

Observations per year:
year
2010    25
2011    25
2012    28
2013    27
2014    29
Name: iso3, dtype: int64


## 7 · Cross-Sectional Dataset (Country Averages 2010–2014)
One row per country — averages of all variables over 2010–2014.


In [93]:
cs = (
    panel_hist
    .groupby(["iso3", "country"])
    .agg(
        years_observed=("year", "count"),
        **{col: (col, "mean") for col in INDICATOR_COLS},
        uk_import_qty_t=("uk_import_qty_t", "mean"),
        log_uk_import_qty_t=("log_uk_import_qty_t", "mean"),
    )
    .reset_index()
    .sort_values("uk_import_qty_t", ascending=False)
    .reset_index(drop=True)
)

print(f"Cross-sectional shape: {cs.shape}  ({cs.shape[0]} countries)")
cs.to_csv(os.path.join(OUT, "crosssection_avg.csv"), index=False)
print(f"Saved -> data/clean/crosssection_avg.csv")

# Display sorted by UK import volume
display_cols = ["iso3", "country", "uk_import_qty_t", "bii", "bii_loss"]
cs[[c for c in display_cols if c in cs.columns]]

Cross-sectional shape: (32, 7)  (32 countries)
Saved -> data/clean/crosssection_avg.csv


,iso3,country,uk_import_qty_t,bii,bii_loss
0,IRL,Ireland,40045.020000,0.419109,0.580891
1,BRA,Brazil,33888.630000,0.767814,0.232186
2,DNK,Denmark,2771.980000,0.449684,0.550316
3,NLD,Netherlands (Kingdom of the),2038.740000,0.597566,0.402434
4,FRA,France,1886.500000,0.618970,0.381030
5,BEL,Belgium,1005.480000,0.640613,0.359387
6,ARG,Argentina,967.080000,0.719480,0.280520
7,DEU,Germany,874.140000,0.682221,0.317779
8,SWE,Sweden,866.700000,0.946333,0.053667
9,URY,Uruguay,587.310000,0.334824,0.665176


## 8 · Extended Panel 2010–2020 (Historical + SSP2 Projections)

To extend coverage beyond 2014 we stitch in **SSP2** ("middle of the road") BIO projections for 2015–2020.

**2015–2020 BIO values are modelled, not observed**

In [94]:
# SSP2 BIO slice for 2015-2020
bio_ssp2 = (
    bio_wide[
        (bio_wide["scenario"] == EXTENSION_SCENARIO) &
        (bio_wide["year"].between(2015, 2020))
    ]
    .drop(columns=["scenario"])
    .copy()
)
print(f"SSP2 BIO slice: {bio_ssp2.shape[0]:,} rows  |  years {bio_ssp2['year'].min()}-{bio_ssp2['year'].max()}")

# Combine: historical 2010-2014 + SSP2 2015-2020
bio_combined = pd.concat([
    bio_hist[bio_hist["year"].between(2010, 2014)],
    bio_ssp2,
], ignore_index=True)

panel_ext = build_panel(bio_combined, fao, "Extended panel 2010-2020 (historical + SSP2)")

# Attach bio_source flag
bio_src = bio_combined[["iso3", "year"]].copy()
bio_src["bio_source"] = np.where(bio_combined["year"] <= 2014, "historical", "ssp2_projected")
panel_ext = panel_ext.merge(bio_src.drop_duplicates(), on=["iso3", "year"], how="left")

panel_ext.to_csv(os.path.join(OUT, "panel_extended_2010_2020.csv"), index=False)
print(f"Saved -> data/clean/panel_extended_2010_2020.csv")
panel_ext.head(10)

SSP2 BIO slice: 1,206 rows  |  years 2015-2020

Extended panel 2010-2020 (historical + SSP2)
  Shape: (292, 7)  |  32 countries x 11 years
Saved -> data/clean/panel_extended_2010_2020.csv


,iso3,country,year,uk_import_qty_t,log_uk_import_qty_t,bii,bii_loss,bio_source
0,ARG,Argentina,2010,1959.00,7.580700,0.725217,0.274783,historical
1,ARG,Argentina,2011,1322.00,7.187657,0.723646,0.276354,historical
2,ARG,Argentina,2012,1182.00,7.075809,0.721324,0.278676,historical
3,ARG,Argentina,2013,314.00,5.752573,0.715945,0.284055,historical
4,ARG,Argentina,2014,58.40,4.084294,0.711267,0.288733,historical
5,ARG,Argentina,2015,23.91,3.215269,0.708774,0.291226,ssp2_projected
6,ARG,Argentina,2016,22.79,3.169265,0.708488,0.291512,ssp2_projected
7,AUS,Australia,2010,7.00,2.079442,0.699317,0.300683,historical
8,AUS,Australia,2011,13.00,2.639057,0.693440,0.306560,historical
9,AUS,Australia,2012,3.00,1.386294,0.694919,0.305081,historical


---
# Part D — Validation & Summary Statistics
## 9 · Descriptive Statistics (Cross-Sectional)

In [95]:
desc_cols = ["uk_import_qty_t", "log_uk_import_qty_t"] + INDICATOR_COLS
cs[desc_cols].describe().round(4)

,uk_import_qty_t,log_uk_import_qty_t,bii,bii_loss
count,32.0000,32.0000,32.0000,32.0000
mean,2691.6838,3.9437,0.6851,0.3149
std,9051.3508,2.9677,0.1357,0.1357
min,0.0200,0.0198,0.3348,0.0480
25%,4.2000,1.5700,0.6133,0.2328
50%,52.1900,3.6846,0.6828,0.3172
75%,868.5600,6.1494,0.7672,0.3867
max,40045.0200,10.5851,0.9520,0.6652


### 9.1 · Correlations with log UK import quantity
We use the **log-transformed** outcome because raw trade volumes are heavily right-skewed.

In [96]:
corr = (
    cs[desc_cols]
    .corr()["log_uk_import_qty_t"]
    .drop(["uk_import_qty_t", "log_uk_import_qty_t"])
    .sort_values(key=abs, ascending=False)
)
print("Pearson correlations with log_uk_import_qty_t:")
print(corr.round(4).to_string())

Pearson correlations with log_uk_import_qty_t:
bii        -0.3752
bii_loss    0.3752


## 10 · Quick OLS Sanity Check (Cross-Sectional)

A preliminary OLS to verify data is usable before moving to full modelling.

- **Model**: `log_import ~ bii_loss`

This tests whether countries with more biodiversity loss export more beef to the UK.
We use **HC3 robust standard errors** to guard against heteroskedasticity from
dominant exporters (Ireland, Brazil).

In [97]:
try:
    import statsmodels.api as sm

    y = cs["log_uk_import_qty_t"].astype(float)

    # -- OLS: bii_loss → log UK beef imports -----------------------------------
    print("OLS: log_uk_import_qty_t ~ bii_loss")
    X = sm.add_constant(cs[["bii_loss"]].astype(float))

    # Standard OLS
    m = sm.OLS(y, X).fit()
    print("\n--- Standard errors ---")
    print(m.summary2().tables[1].round(4))
    print(f"R-sq = {m.rsquared:.4f}  |  Adj-R-sq = {m.rsquared_adj:.4f}  |  N = {int(m.nobs)}")

    # Robust (HC3) standard errors
    m_robust = sm.OLS(y, X).fit(cov_type="HC3")
    print("\n--- Robust (HC3) standard errors ---")
    print(m_robust.summary2().tables[1].round(4))
    print(f"R-sq = {m_robust.rsquared:.4f}  |  Adj-R-sq = {m_robust.rsquared_adj:.4f}  |  N = {int(m_robust.nobs)}")

except ImportError:
    print("statsmodels not installed -- run:  pip install statsmodels")

OLS: log_uk_import_qty_t ~ bii_loss

--- Standard errors ---
           Coef.  Std.Err.       t   P>|t|  [0.025   0.975]
const     1.3601    1.2657  1.0746  0.2911 -1.2247   3.9450
bii_loss  8.2046    3.7003  2.2173  0.0343  0.6477  15.7616
R-sq = 0.1408  |  Adj-R-sq = 0.1122  |  N = 32

--- Robust (HC3) standard errors ---
           Coef.  Std.Err.       z   P>|z|  [0.025   0.975]
const     1.3601    1.4004  0.9713  0.3314 -1.3846   4.1049
bii_loss  8.2046    4.1476  1.9782  0.0479  0.0754  16.3338
R-sq = 0.1408  |  Adj-R-sq = 0.1122  |  N = 32


---
## Summary

**TBD: Final summary after all cleaning and validation steps are complete.**

**Outcome variable (Y):** `uk_import_qty_t` / `log_uk_import_qty_t`

**Predictor (X):** `bii_loss` (= 1 − BII; higher = more biodiversity loss)

---

### Next Steps for Modelling

1. **Use the extended panel (2010–2020)** with fixed/random effects for more statistical power.
2. **Apply robust standard errors** (`HC3`) to guard against heteroskedasticity from dominant exporters.
3. **Run influence diagnostics** (Cook's distance) to check whether results are driven by Ireland or Brazil.
